# Lenguajes y Autómatas II (SCD-1016)
## Sesión 01 · Encuadre, Diagnóstico de Punteros y Metodología IA

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Por qué un compilador necesita mirar la memoria del programa que analiza?
>
> **Objetivo de la sesión:** Entender cómo vive el código fuente en memoria y recorrerlo con punteros sin salirse del buffer, implementando una inversión in-place y auditándola con pruebas automáticas.

**Entregable:** tu función invertir() en artesano_sesion01_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion01.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes una cadena mutable y debes darle vuelta sin reservar memoria nueva. Al terminar, la misma dirección de memoria contiene los caracteres al revés.

Al final de la sesión tendrás: **Una función en C que invierte cualquier cadena en su mismo espacio de memoria y un harness en verde que lo demuestra con seis casos.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 01](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion01.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion01.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion01.h"

void invertir(char *s);
```

- Sin malloc ni arreglos auxiliares: solo dos índices y un char temporal.
- La cadena se modifica en su lugar; no devuelvas una copia.
- invertir(NULL) debe terminar sin caerse.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion01_c.c` junto con `artesano_sesion01_c.c`
> (o `ia_sesion01_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `invertir("HOLA")` | `ALOH` |
| 2 | `invertir("abc")` | `cba` |
| 3 | `invertir("")` | `(cadena vacía)` |
| 4 | `invertir("AB")` | `BA` |
| 5 | `invertir("a")` | `a` |
| 6 | `invertir(NULL)` | `sin crash` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 01 - Inversion de buffer in-place ===
Caso                               esperado:              obtenido:              resultado
[ 1] invertir("HOLA")                   esperado: ALOH         obtenido: ALOH         OK
[ 2] invertir("abc")                    esperado: cba          obtenido: cba          OK
[ 3] invertir("")                       esperado:              obtenido:              OK
[ 4] invertir("AB")                     esperado: BA           obtenido: BA           OK
[ 5] invertir("a")                      esperado: a            obtenido: a            OK
[ 6] invertir(NULL)                     esperado: sin crash    obtenido: sin crash    OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion01.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] El buffer original queda invertido: no hay copia ni malloc.
- [ ] Los seis casos del harness salen OK con código de salida 0.
- [ ] invertir(NULL) y la cadena vacía no provocan fallo de segmentación.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion01_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [8]:
%%writefile api_sesion01.h
/* api_sesion01.h - contrato de la Sesion 01
 * Inversión de buffer in-place
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION01_H
#define API_SESION01_H

void invertir(char *s);

#endif /* API_SESION01_H */


Overwriting api_sesion01.h


In [31]:
%%writefile artesano_sesion01_c.c
/* artesano_sesion01_c.c - Fase 1: El Artesano (sin IA)
 * Inversión de buffer in-place.
 *
 * Implementa lo que declara api_sesion01.h.

#include "api_sesion01.h"
#include <stddef.h>


void invertir(char *s) {
    if ( s == NULL) {
      return;
    }

int inicio = 0;
int fin = 0;
char aux;

while (s[fin] !='\0') {
  fin++;
}

if (fin == 0) {
  return;
}

fin--;

while (inicio < fin) {
aux = s[inicio];
s[inicio] = s[fin];
s[fin] = aux;

inicio++;
fin--;
  }
}


Overwriting artesano_sesion01_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion01_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [32]:
%%writefile ia_sesion01_c.c
/* ia_sesion01_c.c - Fase 2: El Cientifico */

#include "api_sesion01.h"
#include <stddef.h>

void invertir(char *s) {
    if (s == NULL) {
        return;
    }

    int i = 0;
    int j = 0;
    char aux;

    while (s[j] != '\0') {
        j++;
    }

    if (j > 0) {
        j--;

        while (i < j) {
            aux = s[i];
            s[i] = s[j];
            s[j] = aux;
            i++;
            j--;
        }
    }
}

Overwriting ia_sesion01_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [4]:
%%writefile harness_sesion01_c.c
/* harness_sesion01_c.c - auditoria automatica de la Sesion 01
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion01_c.c artesano_sesion01_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion01.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 01 - Inversion de buffer in-place ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { char a[] = "HOLA"; invertir(a); verificar_str("invertir(\"HOLA\")", "ALOH", a); }
    { char b[] = "abc"; invertir(b); verificar_str("invertir(\"abc\")", "cba", b); }
    { char c[] = ""; invertir(c); verificar_str("invertir(\"\")", "", c); }
    { char d[] = "AB"; invertir(d); verificar_str("invertir(\"AB\")", "BA", d); }
    { char e[] = "a"; invertir(e); verificar_str("invertir(\"a\")", "a", e); }
    { invertir(NULL); verificar_str("invertir(NULL)", "sin crash", "sin crash"); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion01.md" "\n");
    return 0;
}


Writing harness_sesion01_c.c


In [9]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion01_c.c artesano_sesion01_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 01 - Inversion de buffer in-place ===
Caso                               esperado:              obtenido:              resultado
[ 1] invertir("HOLA")                   esperado: ALOH         obtenido: ALOH         OK
[ 2] invertir("abc")                    esperado: cba          obtenido: cba          OK
[ 3] invertir("")                       esperado:              obtenido:              OK
[ 4] invertir("AB")                     esperado: BA           obtenido: BA           OK
[ 5] invertir("a")                      esperado: a            obtenido: a            OK
[ 6] invertir(NULL)                     esperado: sin crash    obtenido: sin crash    OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion01.md


In [33]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion01_c.c ia_sesion01_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 01 - Inversion de buffer in-place ===
Caso                               esperado:              obtenido:              resultado
[ 1] invertir("HOLA")                   esperado: ALOH         obtenido: ALOH         OK
[ 2] invertir("abc")                    esperado: cba          obtenido: cba          OK
[ 3] invertir("")                       esperado:              obtenido:              OK
[ 4] invertir("AB")                     esperado: BA           obtenido: BA           OK
[ 5] invertir("a")                      esperado: a            obtenido: a            OK
[ 6] invertir(NULL)                     esperado: sin crash    obtenido: sin crash    OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion01.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion01.md` a `auditorias/auditoria_sesion01.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
